In [2]:
import pandas as pd

In [3]:
makaan = pd.read_csv('makaan_full.csv')   # rename this to whatever your re-downloaded full file is actually called

delhi = pd.read_csv('Indian_housing_Delhi_data.csv')
mumbai = pd.read_csv('Indian_housing_Mumbai_data.csv')
pune = pd.read_csv('Indian_housing_Pune_data.csv')
april = pd.concat([delhi, mumbai, pune], ignore_index=True)

In [4]:
print("MAKAAN")
print(makaan.shape)
print(makaan.columns.tolist())
print(makaan.head(2))

print("\nAPRIL 2024")
print(april.shape)
print(april.columns.tolist())
print(april.head(2))

MAKAAN
(193011, 10)
['seller_type', 'bedroom', 'layout_type', 'property_type', 'locality', 'price', 'area', 'furnish_type', 'bathroom', 'city']
  seller_type  bedroom layout_type     property_type  locality    price  \
0       OWNER      2.0         BHK         Apartment  Bodakdev  20000.0   
1       OWNER      1.0          RK  Studio Apartment   CG Road   7350.0   

     area    furnish_type  bathroom       city  
0  1450.0       Furnished       2.0  Ahmedabad  
1   210.0  Semi-Furnished       1.0  Ahmedabad  

APRIL 2024
(13910, 16)
['house_type', 'house_size', 'location', 'city', 'latitude', 'longitude', 'price', 'currency', 'numBathrooms', 'numBalconies', 'isNegotiable', 'priceSqFt', 'verificationDate', 'description', 'SecurityDeposit', 'Status']
               house_type house_size           location   city   latitude  \
0  1 RK Studio Apartment   400 sq ft            Kalkaji  Delhi  28.545561   
1  1 RK Studio Apartment   400 sq ft  Mansarover Garden  Delhi  28.643259   

   long

In [5]:
makaan = makaan.rename(columns={
    'bedroom': 'bhk',
    'price': 'rent',
    'area': 'size_sqft',
    'furnish_type': 'furnishing',
    'bathroom': 'bathrooms'
})
makaan['source'] = 'makaan'
makaan['latitude'] = None
makaan['longitude'] = None

april = april.rename(columns={
    'location': 'locality',
    'price': 'rent',
    'house_size': 'size_sqft',
    'Status': 'furnishing',
    'numBathrooms': 'bathrooms'
})
april['source'] = 'april2024'
april['bhk'] = april['house_type'].str.extract(r'(\d+)').astype(float)

print(makaan.columns.tolist())
print(april.columns.tolist())

['seller_type', 'bhk', 'layout_type', 'property_type', 'locality', 'rent', 'size_sqft', 'furnishing', 'bathrooms', 'city', 'source', 'latitude', 'longitude']
['house_type', 'size_sqft', 'locality', 'city', 'latitude', 'longitude', 'rent', 'currency', 'bathrooms', 'numBalconies', 'isNegotiable', 'priceSqFt', 'verificationDate', 'description', 'SecurityDeposit', 'furnishing', 'source', 'bhk']


In [6]:
common_cols = ['city', 'locality', 'bhk', 'rent', 'size_sqft', 'furnishing', 'bathrooms', 'latitude', 'longitude', 'source']

makaan_slim = makaan[common_cols]
april_slim = april[common_cols]

combined = pd.concat([makaan_slim, april_slim], ignore_index=True)

print("makaan_slim rows:", len(makaan_slim))
print("april_slim rows:", len(april_slim))
print("combined rows:", len(combined))
combined.head()

makaan_slim rows: 193011
april_slim rows: 13910
combined rows: 206921


,city,locality,bhk,rent,size_sqft,furnishing,bathrooms,latitude,longitude,source
0,Ahmedabad,Bodakdev,2.0,20000.0,1450.0,Furnished,2.0,None,None,makaan
1,Ahmedabad,CG Road,1.0,7350.0,210.0,Semi-Furnished,1.0,None,None,makaan
2,Ahmedabad,Jodhpur,3.0,22000.0,1900.0,Unfurnished,3.0,None,None,makaan
3,Ahmedabad,Sanand,2.0,13000.0,1285.0,Semi-Furnished,2.0,None,None,makaan
4,Ahmedabad,Navrangpura,2.0,18000.0,1600.0,Furnished,2.0,None,None,makaan


In [7]:
unique_localities = combined[combined['latitude'].isna()][['city', 'locality']].drop_duplicates()
print(len(unique_localities))

4211


In [8]:
# ============================================
# CELL 7 (updated) — Geocoding setup with longer timeout
# ============================================
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

geolocator = Nominatim(user_agent="rentora_ai_project", timeout=10)  # was defaulting to 1 second — too short
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=5)

def get_coords(row):
    query = f"{row['locality']}, {row['city']}, India"
    try:
        location = geocode(query)
        if location:
            return pd.Series([location.latitude, location.longitude])
    except Exception as e:
        print(f"Failed: {query} — {e}")
    return pd.Series([None, None])

In [9]:
# CELL 8 — TEST on 5 rows first before committing to the full 70-minute run 
test_batch = unique_localities.head(5).copy()
test_batch[['geo_lat', 'geo_lon']] = test_batch.apply(get_coords, axis=1)
print(test_batch)

        city     locality    geo_lat    geo_lon
0  Ahmedabad     Bodakdev  23.044592  72.517344
1  Ahmedabad      CG Road  23.026011  72.556718
2  Ahmedabad      Jodhpur  23.016927  72.520432
3  Ahmedabad       Sanand  23.023888  72.385148
4  Ahmedabad  Navrangpura  23.036000  72.564343


In [10]:
# ============================================
# CELL 9 (updated) — Geocode in batches, saving progress after each one
# Safer against both interruptions AND intermittent timeouts like the one you just hit
# ============================================
batch_size = 200
results = []

for start in range(0, len(unique_localities), batch_size):
    batch = unique_localities.iloc[start:start+batch_size].copy()
    batch[['geo_lat', 'geo_lon']] = batch.apply(get_coords, axis=1)
    results.append(batch)

    pd.concat(results, ignore_index=True).to_csv('../data/geocoded_localities.csv', index=False)
    print(f"Saved rows {start} to {start+len(batch)} of {len(unique_localities)}")

print("Geocoding fully done and saved.")

KeyboardInterrupt: 

In [11]:
geocoded = pd.read_csv('../data/geocoded_localities.csv')

In [12]:
print(geocoded.shape)
print(geocoded.columns.tolist())
geocoded.head()

(4211, 4)
['city', 'locality', 'geo_lat', 'geo_lon']


,city,locality,geo_lat,geo_lon
0,Ahmedabad,Bodakdev,23.044592,72.517344
1,Ahmedabad,CG Road,23.026011,72.556718
2,Ahmedabad,Jodhpur,23.016927,72.520432
3,Ahmedabad,Sanand,23.023888,72.385148
4,Ahmedabad,Navrangpura,23.036000,72.564343


In [13]:
# ============================================
# Merge geocoded coordinates back into the main combined dataset
# April 2024 rows already had real lat/long; this fills in Makaan's missing ones
# ============================================
combined = combined.merge(
    geocoded[['city', 'locality', 'geo_lat', 'geo_lon']],
    on=['city', 'locality'],
    how='left'
)
combined['latitude'] = combined['latitude'].fillna(combined['geo_lat'])
combined['longitude'] = combined['longitude'].fillna(combined['geo_lon'])
combined = combined.drop(columns=['geo_lat', 'geo_lon'])

print("Still missing coordinates:", combined['latitude'].isna().sum())

Still missing coordinates: 20553


In [14]:
failed = geocoded[geocoded['geo_lat'].isna()]
print(len(failed))
print(failed[['city', 'locality']].head(20))

1309
         city                                locality
8   Ahmedabad                               Shahibuag
12  Ahmedabad                           Saijpur Bogha
13  Ahmedabad                         Jodhpur Village
24  Ahmedabad                                 Ghodsar
27  Ahmedabad                            Keshav Nagar
28  Ahmedabad     Near Nirma University On SG Highway
31  Ahmedabad                              Naranpuraa
36  Ahmedabad                               Satlasana
37  Ahmedabad  Near Vaishno Devi Circle On SG Highway
46  Ahmedabad                                  Zundal
50  Ahmedabad                              Naryanpura
52  Ahmedabad                                Narolgam
53  Ahmedabad                              Usman Pura
54  Ahmedabad                            Nana Chiloda
56  Ahmedabad                           Khodiar Nagar
58  Ahmedabad                       Satellite Society
73  Ahmedabad                                 Kudasan
74  Ahmedabad          

In [15]:
city_coords = {
    'Ahmedabad': (23.0225, 72.5714),
    'Bangalore': (12.9716, 77.5946),
    'Chennai': (13.0827, 80.2707),
    'Delhi': (28.7041, 77.1025),
    'Hyderabad': (17.3850, 78.4867),
    'Kolkata': (22.5726, 88.3639),
    'Mumbai': (19.0760, 72.8777),
    'Pune': (18.5204, 73.8567)
}

def fill_missing(row):
    if pd.isna(row['latitude']) and row['city'] in city_coords:
        return pd.Series(city_coords[row['city']])
    return pd.Series([row['latitude'], row['longitude']])

combined[['latitude', 'longitude']] = combined.apply(fill_missing, axis=1)

print("Still missing coordinates:", combined['latitude'].isna().sum())

Still missing coordinates: 0


In [16]:
combined.to_csv('../data/rentora_combined_raw.csv', index=False)
print("Saved:", combined.shape)

Saved: (206921, 10)
